In [1]:
from sedona.spark import *
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/16 12:02:39 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

In [2]:
database = 'gde_gold'
sedona.sql(f'CREATE DATABASE IF NOT EXISTS org_catalog.{database}')

DataFrame[]

In [3]:
#Verifying if all tables are present
sedona.sql("SHOW TABLES IN org_catalog.gde_silver").show()

+----------+--------------------+-----------+
| namespace|           tableName|isTemporary|
+----------+--------------------+-----------+
|gde_silver|         census_data|      false|
|gde_silver|       homes_buffers|      false|
|gde_silver|  homes_demographics|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver| homes_flood_hazards|      false|
|gde_silver| homes_school_scores|      false|
|gde_silver|homes_seismic_haz...|      false|
|gde_silver|homes_zoning_over...|      false|
|gde_silver|house_sales_ndvi_...|      false|
|gde_silver|  house_sales_silver|      false|
|gde_silver|house_sales_tri_s...|      false|
|gde_silver|house_sales_zonal...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|     roads_proximity|      false|
|gde_silver|         school_data|      false|
|gde_silver|     zoning_overlaps|      false|
+----------+--------------------+-

In [4]:
sedona.sql(f'''
create or replace table org_catalog.{database}.clusters_gold as 
select 
a.sale_id,
a.geometry,
a.sale_price,
a.year_built,
a.sqft,
a.mean_ndvi_500m_radius,
b.avg_school_high_achievement,
c.pw_median_income_proxy,
c.est_total_pop
from org_catalog.gde_silver.house_sales_silver a
join org_catalog.gde_silver.homes_school_scores b using (sale_id)
join org_catalog.gde_silver.homes_demographics c using (sale_id)
''')

DataFrame[]

In [5]:
sedona.sql(f'''
SELECT COUNT(*)
FROM org_catalog.gde_silver.house_sales_silver a
JOIN org_catalog.gde_silver.homes_school_scores b USING (sale_id)
JOIN org_catalog.gde_silver.homes_demographics c USING (sale_id)
WHERE a.sale_price > 50000
AND b.avg_school_high_achievement > 0.10
''').show()

[Stage 10:================================================>         (5 + 1) / 6]

+--------+
|count(1)|
+--------+
|     376|
+--------+



In [6]:
sedona.sql(f''' 
create or replace table org_catalog.{database}.clusters_gold
SELECT
  geometry,
  sale_id,
  sale_price,
  year_built,
  sqft,
  mean_ndvi_500m_radius,
  avg_school_high_achievement,
  pw_median_income_proxy,
  est_total_pop,
  ST_DBSCAN(geometry, 100, 2, true) AS cluster
FROM org_catalog.{database}.clusters_gold
WHERE sale_price > 200000
  AND avg_school_high_achievement > 0.20;
''')

26/03/16 12:05:02 WARN ConnectedComponents$: Returned DataFrame is persistent and materialized!


DataFrame[]

In [7]:
sedona.sql(f'''select cluster.cluster from org_catalog.{database}.clusters_gold''').show()

+----------+
|   cluster|
+----------+
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
|8589934592|
+----------+
only showing top 20 rows


In [8]:
# Confirming table has rows
sedona.sql("SELECT COUNT(*) FROM org_catalog.gde_gold.analytics_gold").show()

+--------+
|count(1)|
+--------+
|     376|
+--------+

